In [1]:
# ============================================================
# 03.06 GAN TRAINING — EXPERIMENT 02
# Cell 1: Environment & Imports
# ============================================================

import sys
import json
import time
import copy
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

print("=" * 70)
print("EXPERIMENT 02 — ENVIRONMENT CHECK")
print("=" * 70)

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device         : {device}")
print("=" * 70)

print("All required imports completed successfully.")

EXPERIMENT 02 — ENVIRONMENT CHECK
Python version : 3.14.4
PyTorch version: 2.13.0+cpu
NumPy version  : 2.5.2
CUDA available : False
Device         : cpu
All required imports completed successfully.


In [2]:
# ============================================================
# Cell 2: Reproducibility Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ensure deterministic behavior where possible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("EXPERIMENT 02 — REPRODUCIBILITY CHECK")
print("=" * 70)

print(f"Random seed    : {SEED}")
print(f"PyTorch seed   : {SEED}")
print(f"NumPy seed     : {SEED}")
print(f"CUDA available : {torch.cuda.is_available()}")

print("=" * 70)
print("Reproducibility configuration completed successfully.")

EXPERIMENT 02 — REPRODUCIBILITY CHECK
Random seed    : 42
PyTorch seed   : 42
NumPy seed     : 42
CUDA available : False
Reproducibility configuration completed successfully.


In [3]:
# ============================================================
# Cell 3: Project Root & Experiment 2 Directory Setup
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — PROJECT PATH SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Step 1: Locate the project root robustly
# ------------------------------------------------------------

current_path = Path.cwd().resolve()

project_root = None

for candidate in [current_path] + list(current_path.parents):
    if (candidate / "06_cross_modal_learning").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'06_cross_modal_learning'."
    )

# ------------------------------------------------------------
# Step 2: Define official Phase 3 paths
# ------------------------------------------------------------

phase_root = project_root / "06_cross_modal_learning"

official_phase3_root = (
    phase_root / "03_cross_modal_learning"
)

gan_training_root = (
    official_phase3_root / "03_06_gan_training"
)

# ------------------------------------------------------------
# Step 3: Create an isolated Experiment 2 directory
# ------------------------------------------------------------

experiment_02_root = (
    gan_training_root / "experiment_02_lambda_adv_0_1"
)

experiment_02_checkpoint_dir = (
    experiment_02_root / "checkpoints"
)

experiment_02_report_dir = (
    experiment_02_root / "reports"
)

experiment_02_figure_dir = (
    experiment_02_root / "figures"
)

# Create only Experiment 2 directories
experiment_02_checkpoint_dir.mkdir(parents=True, exist_ok=True)
experiment_02_report_dir.mkdir(parents=True, exist_ok=True)
experiment_02_figure_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Step 4: Define Experiment 2 output files
# ------------------------------------------------------------

best_generator_path = (
    experiment_02_checkpoint_dir /
    "03_06_gan_experiment_02_best_generator.pt"
)

best_discriminator_path = (
    experiment_02_checkpoint_dir /
    "03_06_gan_experiment_02_best_discriminator.pt"
)

final_generator_path = (
    experiment_02_checkpoint_dir /
    "03_06_gan_experiment_02_final_generator.pt"
)

final_discriminator_path = (
    experiment_02_checkpoint_dir /
    "03_06_gan_experiment_02_final_discriminator.pt"
)

training_history_path = (
    experiment_02_checkpoint_dir /
    "03_06_gan_experiment_02_training_history.pt"
)

configuration_path = (
    experiment_02_root /
    "03_06_gan_experiment_02_configuration.json"
)

# ------------------------------------------------------------
# Step 5: Display and verify paths
# ------------------------------------------------------------

print(f"Project root:")
print(f"  {project_root}")

print(f"\nPhase 3 root:")
print(f"  {official_phase3_root}")

print(f"\nGAN training root:")
print(f"  {gan_training_root}")

print(f"\nExperiment 2 root:")
print(f"  {experiment_02_root}")

print("\nExperiment 2 directories:")
print(f"  Checkpoints : {experiment_02_checkpoint_dir}")
print(f"  Reports     : {experiment_02_report_dir}")
print(f"  Figures     : {experiment_02_figure_dir}")

print("\nExperiment 2 output files:")
print(f"  Best Generator      : {best_generator_path.name}")
print(f"  Best Discriminator  : {best_discriminator_path.name}")
print(f"  Final Generator     : {final_generator_path.name}")
print(f"  Final Discriminator : {final_discriminator_path.name}")
print(f"  Training History    : {training_history_path.name}")
print(f"  Configuration       : {configuration_path.name}")

# ------------------------------------------------------------
# Step 6: Safety check
# ------------------------------------------------------------

if "experiment_02_lambda_adv_0_1" not in str(experiment_02_root):
    raise RuntimeError(
        "Safety check failed: Experiment 2 path is incorrect."
    )

if experiment_02_root == gan_training_root:
    raise RuntimeError(
        "Safety check failed: Experiment 2 must have its own directory."
    )

print("\n" + "=" * 70)
print("PATH SAFETY CHECK PASSED")
print("Experiment 2 has an isolated output directory.")
print("Experiment 1 files will NOT be overwritten.")
print("=" * 70)

EXPERIMENT 02 — PROJECT PATH SETUP
Project root:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 3 root:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning

GAN training root:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training

Experiment 2 root:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_02_lambda_adv_0_1

Experiment 2 directories:
  Checkpoints : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_02_lambda_adv_0_1\checkpoints
  Reports     : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_02_lambda_ad

In [4]:
# ============================================================
# Cell 4: Load Phase 02.08 Representation-Aware Pair Dataset
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — LOADING CROSS-MODAL PAIRS")
print("=" * 70)

# ------------------------------------------------------------
# Phase 02.08 dataset directory
# ------------------------------------------------------------

pair_data_root = (
    phase_root
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)

train_pairs_path = (
    pair_data_root /
    "02_08_train_representation_disjoint_pairs.npz"
)

validation_pairs_path = (
    pair_data_root /
    "02_08_validation_representation_disjoint_pairs.npz"
)

# ------------------------------------------------------------
# Verify files exist before loading
# ------------------------------------------------------------

if not train_pairs_path.exists():
    raise FileNotFoundError(
        f"Training pair file not found:\n{train_pairs_path}"
    )

if not validation_pairs_path.exists():
    raise FileNotFoundError(
        f"Validation pair file not found:\n{validation_pairs_path}"
    )

print("Training pair file found:")
print(f"  {train_pairs_path}")

print("\nValidation pair file found:")
print(f"  {validation_pairs_path}")

# ------------------------------------------------------------
# Load the exact Phase 02.08 arrays
# ------------------------------------------------------------

train_data = np.load(train_pairs_path)
validation_data = np.load(validation_pairs_path)

train_source = train_data["source_embeddings"].astype(np.float32)
train_target = train_data["target_embeddings"].astype(np.float32)
train_labels = train_data["pair_labels"].astype(np.float32)

validation_source = validation_data["source_embeddings"].astype(np.float32)
validation_target = validation_data["target_embeddings"].astype(np.float32)
validation_labels = validation_data["pair_labels"].astype(np.float32)

# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

print("\nLoaded training arrays:")
print(f"  Source embeddings : {train_source.shape}")
print(f"  Target embeddings : {train_target.shape}")
print(f"  Pair labels       : {train_labels.shape}")

print("\nLoaded validation arrays:")
print(f"  Source embeddings : {validation_source.shape}")
print(f"  Target embeddings : {validation_target.shape}")
print(f"  Pair labels       : {validation_labels.shape}")

print("\n" + "=" * 70)
print("PHASE 02.08 DATA LOADED SUCCESSFULLY")
print("=" * 70)

EXPERIMENT 02 — LOADING CROSS-MODAL PAIRS
Training pair file found:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

Validation pair file found:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

Loaded training arrays:
  Source embeddings : (1776, 32)
  Target embeddings : (1776, 32)
  Pair labels       : (1776,)

Loaded validation arrays:
  Source embeddings : (448, 32)
  Target embeddings : (448, 32)
  Pair labels       : (448,)

PHASE 02.08 DATA LOADED SUCCESSFULLY


In [5]:
# ============================================================
# Cell 5: Dataset Integrity & Class-Balance Verification
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — DATASET INTEGRITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify expected sample counts
# ------------------------------------------------------------

EXPECTED_TRAIN_SAMPLES = 1776
EXPECTED_VALIDATION_SAMPLES = 448
EXPECTED_EMBEDDING_DIM = 32

assert train_source.shape == (
    EXPECTED_TRAIN_SAMPLES,
    EXPECTED_EMBEDDING_DIM,
), f"Unexpected training source shape: {train_source.shape}"

assert train_target.shape == (
    EXPECTED_TRAIN_SAMPLES,
    EXPECTED_EMBEDDING_DIM,
), f"Unexpected training target shape: {train_target.shape}"

assert train_labels.shape == (
    EXPECTED_TRAIN_SAMPLES,
), f"Unexpected training label shape: {train_labels.shape}"

assert validation_source.shape == (
    EXPECTED_VALIDATION_SAMPLES,
    EXPECTED_EMBEDDING_DIM,
), f"Unexpected validation source shape: {validation_source.shape}"

assert validation_target.shape == (
    EXPECTED_VALIDATION_SAMPLES,
    EXPECTED_EMBEDDING_DIM,
), f"Unexpected validation target shape: {validation_target.shape}"

assert validation_labels.shape == (
    EXPECTED_VALIDATION_SAMPLES,
), f"Unexpected validation label shape: {validation_labels.shape}"

print("✓ Expected dataset dimensions confirmed")

# ------------------------------------------------------------
# 2. Check for NaN and Inf values
# ------------------------------------------------------------

assert np.isfinite(train_source).all(), \
    "NaN or Inf detected in training source embeddings"

assert np.isfinite(train_target).all(), \
    "NaN or Inf detected in training target embeddings"

assert np.isfinite(train_labels).all(), \
    "NaN or Inf detected in training labels"

assert np.isfinite(validation_source).all(), \
    "NaN or Inf detected in validation source embeddings"

assert np.isfinite(validation_target).all(), \
    "NaN or Inf detected in validation target embeddings"

assert np.isfinite(validation_labels).all(), \
    "NaN or Inf detected in validation labels"

print("✓ No NaN or Inf values detected")

# ------------------------------------------------------------
# 3. Verify labels contain only 0 and 1
# ------------------------------------------------------------

train_unique_labels = np.unique(train_labels)
validation_unique_labels = np.unique(validation_labels)

assert set(train_unique_labels).issubset({0.0, 1.0}), \
    f"Unexpected training labels: {train_unique_labels}"

assert set(validation_unique_labels).issubset({0.0, 1.0}), \
    f"Unexpected validation labels: {validation_unique_labels}"

print("✓ Training and validation labels are binary")

# ------------------------------------------------------------
# 4. Class-balance verification
# ------------------------------------------------------------

train_negative = int(np.sum(train_labels == 0))
train_positive = int(np.sum(train_labels == 1))

validation_negative = int(np.sum(validation_labels == 0))
validation_positive = int(np.sum(validation_labels == 1))

print("\nTraining class distribution:")
print(f"  Negative (0): {train_negative}")
print(f"  Positive (1): {train_positive}")

print("\nValidation class distribution:")
print(f"  Negative (0): {validation_negative}")
print(f"  Positive (1): {validation_positive}")

assert train_negative == train_positive, \
    "Training pairs are not balanced"

assert validation_negative == validation_positive, \
    "Validation pairs are not balanced"

print("\n✓ Training pairs are perfectly balanced")
print("✓ Validation pairs are perfectly balanced")

# ------------------------------------------------------------
# 5. Verify source and target dimensions
# ------------------------------------------------------------

assert train_source.shape[1] == 32
assert train_target.shape[1] == 32
assert validation_source.shape[1] == 32
assert validation_target.shape[1] == 32

print("✓ All embeddings are 32-dimensional")

# ------------------------------------------------------------
# 6. Basic numerical summary
# ------------------------------------------------------------

print("\nNumerical summary:")
print(
    f"  Training source mean/std : "
    f"{train_source.mean():.6f} / {train_source.std():.6f}"
)

print(
    f"  Training target mean/std : "
    f"{train_target.mean():.6f} / {train_target.std():.6f}"
)

print(
    f"  Validation source mean/std : "
    f"{validation_source.mean():.6f} / {validation_source.std():.6f}"
)

print(
    f"  Validation target mean/std : "
    f"{validation_target.mean():.6f} / {validation_target.std():.6f}"
)

print("\n" + "=" * 70)
print("DATASET INTEGRITY CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — DATASET INTEGRITY CHECK
✓ Expected dataset dimensions confirmed
✓ No NaN or Inf values detected
✓ Training and validation labels are binary

Training class distribution:
  Negative (0): 888
  Positive (1): 888

Validation class distribution:
  Negative (0): 224
  Positive (1): 224

✓ Training pairs are perfectly balanced
✓ Validation pairs are perfectly balanced
✓ All embeddings are 32-dimensional

Numerical summary:
  Training source mean/std : 0.002885 / 0.136592
  Training target mean/std : 0.492116 / 2.249762
  Validation source mean/std : 0.004779 / 0.142983
  Validation target mean/std : 0.502920 / 2.223020

DATASET INTEGRITY CHECK PASSED


In [6]:
# ============================================================
# Cell 6: Generator Architecture
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — GENERATOR ARCHITECTURE")
print("=" * 70)


class Generator(nn.Module):
    """
    Generator for cross-modal representation transformation.

    Input:
        Source representation: 32-D

    Output:
        Generated target-like representation: 32-D

    Architecture:
        32 → 64 → 128 → 64 → 32
    """

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32)
        )

    def forward(self, source):
        return self.network(source)


# ------------------------------------------------------------
# Create Generator
# ------------------------------------------------------------

generator = Generator().to(device)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

generator_total_params = sum(
    parameter.numel()
    for parameter in generator.parameters()
)

generator_trainable_params = sum(
    parameter.numel()
    for parameter in generator.parameters()
    if parameter.requires_grad
)

print("\nGenerator architecture:")
print(generator)

print("\nParameter count:")
print(f"  Total parameters     : {generator_total_params:,}")
print(f"  Trainable parameters : {generator_trainable_params:,}")

# ------------------------------------------------------------
# Expected parameter-count verification
# ------------------------------------------------------------

EXPECTED_GENERATOR_PARAMS = 20_768

assert generator_total_params == EXPECTED_GENERATOR_PARAMS, (
    f"Unexpected Generator parameter count: "
    f"{generator_total_params}"
)

assert generator_trainable_params == EXPECTED_GENERATOR_PARAMS, (
    "Generator parameters are not all trainable."
)

# ------------------------------------------------------------
# Forward-pass test
# ------------------------------------------------------------

test_source = torch.from_numpy(
    train_source[:32]
).to(device)

with torch.no_grad():
    test_generated_target = generator(test_source)

print("\nForward-pass test:")
print(f"  Input shape  : {test_source.shape}")
print(f"  Output shape : {test_generated_target.shape}")

assert test_generated_target.shape == (32, 32), (
    f"Unexpected Generator output shape: "
    f"{test_generated_target.shape}"
)

assert torch.isfinite(test_generated_target).all(), (
    "Generator produced NaN or Inf values."
)

print("\n✓ Generator parameter count verified")
print("✓ Generator output dimension verified")
print("✓ Generator output contains only finite values")

print("\n" + "=" * 70)
print("GENERATOR ARCHITECTURE CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — GENERATOR ARCHITECTURE

Generator architecture:
Generator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
  )
)

Parameter count:
  Total parameters     : 20,768
  Trainable parameters : 20,768

Forward-pass test:
  Input shape  : torch.Size([32, 32])
  Output shape : torch.Size([32, 32])

✓ Generator parameter count verified
✓ Generator output dimension verified
✓ Generator output contains only finite values

GENERATOR ARCHITECTURE CHECK PASSED


In [7]:
# ============================================================
# Cell 7: Discriminator Architecture
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — DISCRIMINATOR ARCHITECTURE")
print("=" * 70)


class Discriminator(nn.Module):
    """
    Discriminator for distinguishing real and generated
    target-domain representations.

    Input:
        Target representation: 32-D

    Output:
        Single logit representing real/fake prediction.

    Architecture:
        32 → 64 → 32 → 1
    """

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, target):
        return self.network(target).squeeze(-1)


# ------------------------------------------------------------
# Create Discriminator
# ------------------------------------------------------------

discriminator = Discriminator().to(device)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

discriminator_total_params = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
)

discriminator_trainable_params = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
    if parameter.requires_grad
)

print("\nDiscriminator architecture:")
print(discriminator)

print("\nParameter count:")
print(f"  Total parameters     : {discriminator_total_params:,}")
print(f"  Trainable parameters : {discriminator_trainable_params:,}")

# ------------------------------------------------------------
# Expected parameter-count verification
# ------------------------------------------------------------

EXPECTED_DISCRIMINATOR_PARAMS = 4_225

assert discriminator_total_params == EXPECTED_DISCRIMINATOR_PARAMS, (
    f"Unexpected Discriminator parameter count: "
    f"{discriminator_total_params}"
)

assert discriminator_trainable_params == EXPECTED_DISCRIMINATOR_PARAMS, (
    "Discriminator parameters are not all trainable."
)

# ------------------------------------------------------------
# Forward-pass test
# ------------------------------------------------------------

test_target = torch.from_numpy(
    train_target[:32]
).to(device)

with torch.no_grad():
    test_discriminator_output = discriminator(test_target)

print("\nForward-pass test:")
print(f"  Input shape  : {test_target.shape}")
print(f"  Output shape : {test_discriminator_output.shape}")

assert test_discriminator_output.shape == (32,), (
    f"Unexpected Discriminator output shape: "
    f"{test_discriminator_output.shape}"
)

assert torch.isfinite(test_discriminator_output).all(), (
    "Discriminator produced NaN or Inf values."
)

print("\n✓ Discriminator parameter count verified")
print("✓ Discriminator output dimension verified")
print("✓ Discriminator output contains only finite values")

print("\n" + "=" * 70)
print("DISCRIMINATOR ARCHITECTURE CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — DISCRIMINATOR ARCHITECTURE

Discriminator architecture:
Discriminator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Parameter count:
  Total parameters     : 4,225
  Trainable parameters : 4,225

Forward-pass test:
  Input shape  : torch.Size([32, 32])
  Output shape : torch.Size([32])

✓ Discriminator parameter count verified
✓ Discriminator output dimension verified
✓ Discriminator output contains only finite values

DISCRIMINATOR ARCHITECTURE CHECK PASSED


In [8]:
# ============================================================
# Cell 8: Load Official Frozen Baseline Checkpoint
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — LOADING OFFICIAL BASELINE")
print("=" * 70)

# ------------------------------------------------------------
# Official baseline checkpoint path
# ------------------------------------------------------------

baseline_checkpoint_path = (
    official_phase3_root
    / "03_04_baseline_training"
    / "checkpoints"
    / "03_04_baseline_best_model.pt"
)

# ------------------------------------------------------------
# Verify checkpoint exists
# ------------------------------------------------------------

if not baseline_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Official baseline checkpoint not found:\n"
        f"{baseline_checkpoint_path}"
    )

print("Official baseline checkpoint found:")
print(f"  {baseline_checkpoint_path}")

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

baseline_checkpoint = torch.load(
    baseline_checkpoint_path,
    map_location=device
)

print("\nCheckpoint loaded successfully.")

# ------------------------------------------------------------
# Inspect checkpoint structure
# ------------------------------------------------------------

if isinstance(baseline_checkpoint, dict):
    print("\nCheckpoint contents:")
    for key in baseline_checkpoint.keys():
        print(f"  - {key}")
else:
    print(
        "\nCheckpoint is a direct state dictionary "
        "rather than a metadata dictionary."
    )

print("\n" + "=" * 70)
print("BASELINE CHECKPOINT LOAD PASSED")
print("=" * 70)

EXPERIMENT 02 — LOADING OFFICIAL BASELINE
Official baseline checkpoint found:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training\checkpoints\03_04_baseline_best_model.pt

Checkpoint loaded successfully.

Checkpoint contents:
  - epoch
  - model_state_dict
  - optimizer_state_dict
  - best_val_roc_auc
  - seed
  - batch_size
  - learning_rate
  - num_epochs

BASELINE CHECKPOINT LOAD PASSED


In [9]:
# ============================================================
# Cell 9: Recreate and Load Frozen Baseline Model
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — FROZEN BASELINE MODEL")
print("=" * 70)


class BaselineCrossModalModel(nn.Module):
    """
    Official baseline cross-modal relationship model.

    Input construction:
        [S, T, |S-T|, S*T]

    Dimensions:
        32 + 32 + 32 + 32 = 128

    Architecture:
        128 → 128 → 64 → 1
    """

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, source, target):

        absolute_difference = torch.abs(source - target)
        elementwise_product = source * target

        combined = torch.cat(
            [
                source,
                target,
                absolute_difference,
                elementwise_product
            ],
            dim=1
        )

        return self.classifier(combined).squeeze(-1)


# ------------------------------------------------------------
# Create baseline model
# ------------------------------------------------------------

baseline_model = BaselineCrossModalModel().to(device)

# ------------------------------------------------------------
# Verify parameter count
# ------------------------------------------------------------

baseline_total_params = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)

EXPECTED_BASELINE_PARAMS = 24_833

assert baseline_total_params == EXPECTED_BASELINE_PARAMS, (
    f"Unexpected baseline parameter count: "
    f"{baseline_total_params}"
)

print(f"Baseline parameters: {baseline_total_params:,}")

# ------------------------------------------------------------
# Load official checkpoint weights
# ------------------------------------------------------------

if "model_state_dict" not in baseline_checkpoint:
    raise KeyError(
        "The checkpoint does not contain 'model_state_dict'."
    )

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# Freeze baseline
# ------------------------------------------------------------

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

baseline_model.eval()

# ------------------------------------------------------------
# Verify frozen state
# ------------------------------------------------------------

baseline_trainable_params = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

assert baseline_trainable_params == 0, (
    "Baseline still contains trainable parameters."
)

# ------------------------------------------------------------
# Forward-pass verification
# ------------------------------------------------------------

test_source = torch.from_numpy(
    train_source[:32]
).to(device)

test_target = torch.from_numpy(
    train_target[:32]
).to(device)

with torch.no_grad():
    baseline_output = baseline_model(
        test_source,
        test_target
    )

print("\nForward-pass test:")
print(f"  Source shape : {test_source.shape}")
print(f"  Target shape : {test_target.shape}")
print(f"  Output shape : {baseline_output.shape}")

assert baseline_output.shape == (32,), (
    f"Unexpected baseline output shape: "
    f"{baseline_output.shape}"
)

assert torch.isfinite(baseline_output).all(), (
    "Baseline produced NaN or Inf values."
)

print("\nBaseline status:")
print(f"  Total parameters     : {baseline_total_params:,}")
print(f"  Trainable parameters : {baseline_trainable_params:,}")
print(f"  Evaluation mode      : {not baseline_model.training}")

print("\n✓ Official checkpoint loaded")
print("✓ Architecture verified")
print("✓ Baseline completely frozen")
print("✓ Forward pass verified")

print("\n" + "=" * 70)
print("FROZEN BASELINE CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — FROZEN BASELINE MODEL
Baseline parameters: 24,833

Forward-pass test:
  Source shape : torch.Size([32, 32])
  Target shape : torch.Size([32, 32])
  Output shape : torch.Size([32])

Baseline status:
  Total parameters     : 24,833
  Trainable parameters : 0
  Evaluation mode      : True

✓ Official checkpoint loaded
✓ Architecture verified
✓ Baseline completely frozen
✓ Forward pass verified

FROZEN BASELINE CHECK PASSED


In [10]:
# ============================================================
# Cell 9: Recreate and Load Frozen Baseline Model
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — FROZEN BASELINE MODEL")
print("=" * 70)


class BaselineCrossModalModel(nn.Module):
    """
    Official baseline cross-modal relationship model.

    Input construction:
        [S, T, |S-T|, S*T]

    Dimensions:
        32 + 32 + 32 + 32 = 128

    Architecture:
        128 → 128 → 64 → 1
    """

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, source, target):

        absolute_difference = torch.abs(source - target)
        elementwise_product = source * target

        combined = torch.cat(
            [
                source,
                target,
                absolute_difference,
                elementwise_product
            ],
            dim=1
        )

        return self.classifier(combined).squeeze(-1)


# ------------------------------------------------------------
# Create baseline model
# ------------------------------------------------------------

baseline_model = BaselineCrossModalModel().to(device)

# ------------------------------------------------------------
# Verify parameter count
# ------------------------------------------------------------

baseline_total_params = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)

EXPECTED_BASELINE_PARAMS = 24_833

assert baseline_total_params == EXPECTED_BASELINE_PARAMS, (
    f"Unexpected baseline parameter count: "
    f"{baseline_total_params}"
)

print(f"Baseline parameters: {baseline_total_params:,}")

# ------------------------------------------------------------
# Load official checkpoint weights
# ------------------------------------------------------------

if "model_state_dict" not in baseline_checkpoint:
    raise KeyError(
        "The checkpoint does not contain 'model_state_dict'."
    )

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# Freeze baseline
# ------------------------------------------------------------

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

baseline_model.eval()

# ------------------------------------------------------------
# Verify frozen state
# ------------------------------------------------------------

baseline_trainable_params = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

assert baseline_trainable_params == 0, (
    "Baseline still contains trainable parameters."
)

# ------------------------------------------------------------
# Forward-pass verification
# ------------------------------------------------------------

test_source = torch.from_numpy(
    train_source[:32]
).to(device)

test_target = torch.from_numpy(
    train_target[:32]
).to(device)

with torch.no_grad():
    baseline_output = baseline_model(
        test_source,
        test_target
    )

print("\nForward-pass test:")
print(f"  Source shape : {test_source.shape}")
print(f"  Target shape : {test_target.shape}")
print(f"  Output shape : {baseline_output.shape}")

assert baseline_output.shape == (32,), (
    f"Unexpected baseline output shape: "
    f"{baseline_output.shape}"
)

assert torch.isfinite(baseline_output).all(), (
    "Baseline produced NaN or Inf values."
)

print("\nBaseline status:")
print(f"  Total parameters     : {baseline_total_params:,}")
print(f"  Trainable parameters : {baseline_trainable_params:,}")
print(f"  Evaluation mode      : {not baseline_model.training}")

print("\n✓ Official checkpoint loaded")
print("✓ Architecture verified")
print("✓ Baseline completely frozen")
print("✓ Forward pass verified")

print("\n" + "=" * 70)
print("FROZEN BASELINE CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — FROZEN BASELINE MODEL
Baseline parameters: 24,833

Forward-pass test:
  Source shape : torch.Size([32, 32])
  Target shape : torch.Size([32, 32])
  Output shape : torch.Size([32])

Baseline status:
  Total parameters     : 24,833
  Trainable parameters : 0
  Evaluation mode      : True

✓ Official checkpoint loaded
✓ Architecture verified
✓ Baseline completely frozen
✓ Forward pass verified

FROZEN BASELINE CHECK PASSED


In [11]:
# ============================================================
# Cell 10: GAN Model Safety Audit
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — MODEL SAFETY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Ensure correct modes
# ------------------------------------------------------------

generator.train()
discriminator.train()
baseline_model.eval()

# ------------------------------------------------------------
# Ensure Generator is trainable
# ------------------------------------------------------------

for parameter in generator.parameters():
    parameter.requires_grad = True

# ------------------------------------------------------------
# Ensure Discriminator is trainable
# ------------------------------------------------------------

for parameter in discriminator.parameters():
    parameter.requires_grad = True

# ------------------------------------------------------------
# Ensure Baseline is frozen
# ------------------------------------------------------------

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------

generator_trainable = sum(
    parameter.numel()
    for parameter in generator.parameters()
    if parameter.requires_grad
)

discriminator_trainable = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
    if parameter.requires_grad
)

baseline_trainable = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert generator_trainable == 20_768, (
    f"Unexpected Generator trainable parameters: "
    f"{generator_trainable}"
)

assert discriminator_trainable == 4_225, (
    f"Unexpected Discriminator trainable parameters: "
    f"{discriminator_trainable}"
)

assert baseline_trainable == 0, (
    f"Baseline must be frozen, but has "
    f"{baseline_trainable} trainable parameters."
)

assert generator.training is True
assert discriminator.training is True
assert baseline_model.training is False

# ------------------------------------------------------------
# Display audit
# ------------------------------------------------------------

print("\nModel training status:")

print(
    f"  Generator:"
    f"\n    Trainable parameters : {generator_trainable:,}"
    f"\n    Training mode        : {generator.training}"
)

print(
    f"\n  Discriminator:"
    f"\n    Trainable parameters : {discriminator_trainable:,}"
    f"\n    Training mode        : {discriminator.training}"
)

print(
    f"\n  Frozen Baseline:"
    f"\n    Trainable parameters : {baseline_trainable:,}"
    f"\n    Evaluation mode      : {not baseline_model.training}"
)

print("\n" + "=" * 70)
print("MODEL SAFETY AUDIT PASSED")
print("=" * 70)
print("Generator      → TRAINABLE")
print("Discriminator  → TRAINABLE")
print("Baseline       → FROZEN")
print("=" * 70)

EXPERIMENT 02 — MODEL SAFETY AUDIT

Model training status:
  Generator:
    Trainable parameters : 20,768
    Training mode        : True

  Discriminator:
    Trainable parameters : 4,225
    Training mode        : True

  Frozen Baseline:
    Trainable parameters : 0
    Evaluation mode      : True

MODEL SAFETY AUDIT PASSED
Generator      → TRAINABLE
Discriminator  → TRAINABLE
Baseline       → FROZEN


In [12]:
# ============================================================
# Cell 11: Cross-Modal GAN Dataset
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — DATASET OBJECT")
print("=" * 70)


class CrossModalGANDataset(Dataset):
    """
    Dataset for unpaired, class-consistent cross-modal GAN training.

    Each sample contains:
        source representation : 32-D
        target representation : 32-D
        pair label             : 0 or 1
    """

    def __init__(self, source_embeddings, target_embeddings, labels):

        assert len(source_embeddings) == len(target_embeddings)
        assert len(source_embeddings) == len(labels)

        self.source_embeddings = torch.from_numpy(
            source_embeddings
        ).float()

        self.target_embeddings = torch.from_numpy(
            target_embeddings
        ).float()

        self.labels = torch.from_numpy(
            labels
        ).float()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        source = self.source_embeddings[index]
        target = self.target_embeddings[index]
        label = self.labels[index]

        return source, target, label


# ------------------------------------------------------------
# Create training and validation datasets
# ------------------------------------------------------------

train_dataset = CrossModalGANDataset(
    train_source,
    train_target,
    train_labels
)

validation_dataset = CrossModalGANDataset(
    validation_source,
    validation_target,
    validation_labels
)

# ------------------------------------------------------------
# Verify dataset lengths
# ------------------------------------------------------------

print(f"Training dataset samples   : {len(train_dataset):,}")
print(f"Validation dataset samples : {len(validation_dataset):,}")

assert len(train_dataset) == 1776
assert len(validation_dataset) == 448

# ------------------------------------------------------------
# Inspect one sample
# ------------------------------------------------------------

sample_source, sample_target, sample_label = train_dataset[0]

print("\nSingle-sample shapes:")
print(f"  Source : {sample_source.shape}")
print(f"  Target : {sample_target.shape}")
print(f"  Label  : {sample_label.shape}")

print(f"\nSample label value: {sample_label.item()}")

assert sample_source.shape == (32,)
assert sample_target.shape == (32,)
assert sample_label.ndim == 0
assert sample_label.item() in (0.0, 1.0)

print("\n✓ Dataset object created successfully")
print("✓ Training and validation lengths verified")
print("✓ Individual sample structure verified")

print("\n" + "=" * 70)
print("DATASET OBJECT CHECK PASSED")
print("=" * 70)

EXPERIMENT 02 — DATASET OBJECT
Training dataset samples   : 1,776
Validation dataset samples : 448

Single-sample shapes:
  Source : torch.Size([32])
  Target : torch.Size([32])
  Label  : torch.Size([])

Sample label value: 1.0

✓ Dataset object created successfully
✓ Training and validation lengths verified
✓ Individual sample structure verified

DATASET OBJECT CHECK PASSED


In [13]:
# ============================================================
# Cell 12: Training and Validation DataLoaders
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — DATALOADER SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 32

# Dedicated generator for reproducible training shuffling
train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator,
    drop_last=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print(f"Batch size              : {BATCH_SIZE}")
print(f"Training batches        : {len(train_loader)}")
print(f"Validation batches      : {len(validation_loader)}")
print(f"Training shuffle        : True")
print(f"Validation shuffle      : False")
print(f"Shuffle seed            : {SEED}")

# ------------------------------------------------------------
# Inspect one training batch
# ------------------------------------------------------------

batch_source, batch_target, batch_labels = next(
    iter(train_loader)
)

print("\nTraining batch:")
print(f"  Source shape : {batch_source.shape}")
print(f"  Target shape : {batch_target.shape}")
print(f"  Label shape  : {batch_labels.shape}")

# ------------------------------------------------------------
# Verify batch dimensions
# ------------------------------------------------------------

assert batch_source.shape == (32, 32)
assert batch_target.shape == (32, 32)
assert batch_labels.shape == (32,)

assert torch.isfinite(batch_source).all()
assert torch.isfinite(batch_target).all()
assert torch.isfinite(batch_labels).all()

assert torch.all(
    (batch_labels == 0) | (batch_labels == 1)
)

# ------------------------------------------------------------
# Verify expected number of batches
# ------------------------------------------------------------

assert len(train_loader) == 56
assert len(validation_loader) == 14

print("\n✓ Batch dimensions verified")
print("✓ Batch values are finite")
print("✓ Batch labels are binary")
print("✓ Expected number of batches verified")

print("\n" + "=" * 70)
print("DATALOADER SETUP PASSED")
print("=" * 70)

EXPERIMENT 02 — DATALOADER SETUP
Batch size              : 32
Training batches        : 56
Validation batches      : 14
Training shuffle        : True
Validation shuffle      : False
Shuffle seed            : 42

Training batch:
  Source shape : torch.Size([32, 32])
  Target shape : torch.Size([32, 32])
  Label shape  : torch.Size([32])

✓ Batch dimensions verified
✓ Batch values are finite
✓ Batch labels are binary
✓ Expected number of batches verified

DATALOADER SETUP PASSED


In [14]:
# ============================================================
# Cell 13: GAN Loss Functions
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — LOSS FUNCTION SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Adversarial loss
# ------------------------------------------------------------

adversarial_loss_fn = nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# Semantic relationship loss
# ------------------------------------------------------------

semantic_loss_fn = nn.BCEWithLogitsLoss()

print("Adversarial loss : BCEWithLogitsLoss")
print("Semantic loss    : BCEWithLogitsLoss")

print("\nLoss functions successfully initialized.")

print("\n" + "=" * 70)
print("LOSS FUNCTION SETUP PASSED")
print("=" * 70)

EXPERIMENT 02 — LOSS FUNCTION SETUP
Adversarial loss : BCEWithLogitsLoss
Semantic loss    : BCEWithLogitsLoss

Loss functions successfully initialized.

LOSS FUNCTION SETUP PASSED


In [15]:
# ============================================================
# Cell 14: Experiment 02 Loss Weights
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — LOSS WEIGHT CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Controlled Experiment 02 change
# ------------------------------------------------------------

ADVERSARIAL_LOSS_WEIGHT = 0.1
SEMANTIC_LOSS_WEIGHT = 1.0

# ------------------------------------------------------------
# Generator objective
#
# L_G = λ_adv * L_adv + λ_semantic * L_semantic
# ------------------------------------------------------------

print("Generator loss configuration:")
print(
    f"  Adversarial loss weight : "
    f"{ADVERSARIAL_LOSS_WEIGHT}"
)

print(
    f"  Semantic loss weight    : "
    f"{SEMANTIC_LOSS_WEIGHT}"
)

print("\nGenerator objective:")
print(
    "  L_G = "
    f"{ADVERSARIAL_LOSS_WEIGHT} × L_adv + "
    f"{SEMANTIC_LOSS_WEIGHT} × L_semantic"
)

# ------------------------------------------------------------
# Verify the controlled change
# ------------------------------------------------------------

assert ADVERSARIAL_LOSS_WEIGHT == 0.1
assert SEMANTIC_LOSS_WEIGHT == 1.0

print("\nControlled change from Experiment 01:")
print("  Adversarial weight: 1.0 → 0.1")
print("  Semantic weight   : 1.0 → 1.0")

print("\n✓ Adversarial contribution reduced by 10×")
print("✓ Semantic contribution unchanged")
print("✓ Architecture remains unchanged")
print("✓ Dataset remains unchanged")

print("\n" + "=" * 70)
print("EXPERIMENT 02 LOSS CONFIGURATION PASSED")
print("=" * 70)

EXPERIMENT 02 — LOSS WEIGHT CONFIGURATION
Generator loss configuration:
  Adversarial loss weight : 0.1
  Semantic loss weight    : 1.0

Generator objective:
  L_G = 0.1 × L_adv + 1.0 × L_semantic

Controlled change from Experiment 01:
  Adversarial weight: 1.0 → 0.1
  Semantic weight   : 1.0 → 1.0

✓ Adversarial contribution reduced by 10×
✓ Semantic contribution unchanged
✓ Architecture remains unchanged
✓ Dataset remains unchanged

EXPERIMENT 02 LOSS CONFIGURATION PASSED


In [16]:
# ============================================================
# Cell 15: Generator and Discriminator Optimizers
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — OPTIMIZER SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------

GAN_LEARNING_RATE = 0.001

# ------------------------------------------------------------
# Generator optimizer
# ------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=GAN_LEARNING_RATE
)

# ------------------------------------------------------------
# Discriminator optimizer
# ------------------------------------------------------------

discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=GAN_LEARNING_RATE
)

# ------------------------------------------------------------
# Verify optimized parameters
# ------------------------------------------------------------

generator_optimized_params = sum(
    parameter.numel()
    for group in generator_optimizer.param_groups
    for parameter in group["params"]
)

discriminator_optimized_params = sum(
    parameter.numel()
    for group in discriminator_optimizer.param_groups
    for parameter in group["params"]
)

assert generator_optimized_params == 20_768
assert discriminator_optimized_params == 4_225

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print(f"Optimizer               : Adam")
print(f"Generator learning rate : {GAN_LEARNING_RATE}")
print(f"Discriminator LR        : {GAN_LEARNING_RATE}")

print("\nParameters being optimized:")
print(f"  Generator     : {generator_optimized_params:,}")
print(f"  Discriminator : {discriminator_optimized_params:,}")
print(f"  Baseline      : 0 (frozen)")

print("\n✓ Adam optimizers initialized")
print("✓ Generator optimizer parameters verified")
print("✓ Discriminator optimizer parameters verified")
print("✓ Frozen baseline is not optimized")

print("\n" + "=" * 70)
print("OPTIMIZER SETUP PASSED")
print("=" * 70)

EXPERIMENT 02 — OPTIMIZER SETUP
Optimizer               : Adam
Generator learning rate : 0.001
Discriminator LR        : 0.001

Parameters being optimized:
  Generator     : 20,768
  Discriminator : 4,225
  Baseline      : 0 (frozen)

✓ Adam optimizers initialized
✓ Generator optimizer parameters verified
✓ Discriminator optimizer parameters verified
✓ Frozen baseline is not optimized

OPTIMIZER SETUP PASSED


In [17]:
# ============================================================
# Cell 16: Single-Batch GAN Training Sanity Test
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — SINGLE-BATCH TRAINING TEST")
print("=" * 70)

# ------------------------------------------------------------
# Get one training batch
# ------------------------------------------------------------

batch_source, batch_real_target, batch_labels = next(
    iter(train_loader)
)

batch_source = batch_source.to(device)
batch_real_target = batch_real_target.to(device)
batch_labels = batch_labels.to(device)

print("Batch shapes:")
print(f"  Source      : {batch_source.shape}")
print(f"  Real target : {batch_real_target.shape}")
print(f"  Labels      : {batch_labels.shape}")

# ------------------------------------------------------------
# Save copies of parameters before update
# ------------------------------------------------------------

generator_before = {
    name: parameter.detach().clone()
    for name, parameter in generator.named_parameters()
}

discriminator_before = {
    name: parameter.detach().clone()
    for name, parameter in discriminator.named_parameters()
}

baseline_before = {
    name: parameter.detach().clone()
    for name, parameter in baseline_model.named_parameters()
}

# ============================================================
# 1. DISCRIMINATOR UPDATE
# ============================================================

discriminator_optimizer.zero_grad()

# Real target prediction
real_logits = discriminator(batch_real_target)

# Real labels = 1
real_labels = torch.ones_like(real_logits)

discriminator_real_loss = adversarial_loss_fn(
    real_logits,
    real_labels
)

# Generate fake target
generated_target = generator(batch_source)

# Detach so Generator is NOT updated during D step
fake_logits = discriminator(
    generated_target.detach()
)

# Fake labels = 0
fake_labels = torch.zeros_like(fake_logits)

discriminator_fake_loss = adversarial_loss_fn(
    fake_logits,
    fake_labels
)

# Total discriminator loss
discriminator_loss = (
    discriminator_real_loss +
    discriminator_fake_loss
)

discriminator_loss.backward()

# ------------------------------------------------------------
# Check discriminator gradients
# ------------------------------------------------------------

discriminator_gradients_finite = all(
    parameter.grad is None or
    torch.isfinite(parameter.grad).all()
    for parameter in discriminator.parameters()
)

assert discriminator_gradients_finite, (
    "Non-finite discriminator gradients detected."
)

discriminator_optimizer.step()

# ============================================================
# 2. GENERATOR UPDATE
# ============================================================

generator_optimizer.zero_grad()

# Generate target again
generated_target = generator(batch_source)

# ------------------------------------------------------------
# Adversarial Generator loss
# ------------------------------------------------------------

generated_logits = discriminator(generated_target)

generator_real_labels = torch.ones_like(
    generated_logits
)

generator_adversarial_loss = adversarial_loss_fn(
    generated_logits,
    generator_real_labels
)

# ------------------------------------------------------------
# Semantic relationship loss
# ------------------------------------------------------------

semantic_logits = baseline_model(
    batch_source,
    generated_target
)

semantic_loss = semantic_loss_fn(
    semantic_logits,
    batch_labels
)

# ------------------------------------------------------------
# Experiment 02 weighted Generator objective
# ------------------------------------------------------------

generator_loss = (
    ADVERSARIAL_LOSS_WEIGHT *
    generator_adversarial_loss
    +
    SEMANTIC_LOSS_WEIGHT *
    semantic_loss
)

generator_loss.backward()

# ------------------------------------------------------------
# Check Generator gradients
# ------------------------------------------------------------

generator_gradients_finite = all(
    parameter.grad is None or
    torch.isfinite(parameter.grad).all()
    for parameter in generator.parameters()
)

assert generator_gradients_finite, (
    "Non-finite Generator gradients detected."
)

generator_optimizer.step()

# ============================================================
# 3. BASELINE FREEZE VERIFICATION
# ============================================================

baseline_unchanged = True

for name, parameter in baseline_model.named_parameters():

    if not torch.equal(
        baseline_before[name],
        parameter.detach()
    ):
        baseline_unchanged = False
        break

assert baseline_unchanged, (
    "CRITICAL: Frozen baseline parameters changed!"
)

# ============================================================
# 4. CHECK PARAMETER UPDATES
# ============================================================

generator_changed = any(
    not torch.equal(
        generator_before[name],
        parameter.detach()
    )
    for name, parameter in generator.named_parameters()
)

discriminator_changed = any(
    not torch.equal(
        discriminator_before[name],
        parameter.detach()
    )
    for name, parameter in discriminator.named_parameters()
)

assert generator_changed, (
    "Generator parameters did not change."
)

assert discriminator_changed, (
    "Discriminator parameters did not change."
)

# ============================================================
# 5. FINITE-LOSS CHECK
# ============================================================

all_losses_finite = all(
    torch.isfinite(loss).item()
    for loss in [
        discriminator_real_loss,
        discriminator_fake_loss,
        discriminator_loss,
        generator_adversarial_loss,
        semantic_loss,
        generator_loss
    ]
)

assert all_losses_finite, (
    "One or more losses are non-finite."
)

# ============================================================
# RESULTS
# ============================================================

print("\nLoss values:")

print(
    f"  Discriminator real loss : "
    f"{discriminator_real_loss.item():.10f}"
)

print(
    f"  Discriminator fake loss : "
    f"{discriminator_fake_loss.item():.10f}"
)

print(
    f"  Discriminator total     : "
    f"{discriminator_loss.item():.10f}"
)

print(
    f"  Generator adversarial   : "
    f"{generator_adversarial_loss.item():.10f}"
)

print(
    f"  Generator semantic      : "
    f"{semantic_loss.item():.10f}"
)

print(
    f"  Generator total         : "
    f"{generator_loss.item():.10f}"
)

print("\nExperiment 02 weighting:")
print(
    f"  {ADVERSARIAL_LOSS_WEIGHT} × "
    f"L_adv + "
    f"{SEMANTIC_LOSS_WEIGHT} × "
    f"L_semantic"
)

print("\nUpdate verification:")
print(f"  Generator changed     : {generator_changed}")
print(f"  Discriminator changed : {discriminator_changed}")
print(f"  Baseline unchanged    : {baseline_unchanged}")

print("\nGradient verification:")
print(
    f"  Generator gradients finite     : "
    f"{generator_gradients_finite}"
)

print(
    f"  Discriminator gradients finite : "
    f"{discriminator_gradients_finite}"
)

print(f"\nAll losses finite : {all_losses_finite}")

print("\n" + "=" * 70)
print("SINGLE-BATCH TRAINING TEST PASSED")
print("=" * 70)

EXPERIMENT 02 — SINGLE-BATCH TRAINING TEST
Batch shapes:
  Source      : torch.Size([32, 32])
  Real target : torch.Size([32, 32])
  Labels      : torch.Size([32])

Loss values:
  Discriminator real loss : 0.6883126497
  Discriminator fake loss : 0.7772378325
  Discriminator total     : 1.4655504227
  Generator adversarial   : 0.6165086031
  Generator semantic      : 0.6348801851
  Generator total         : 0.6965310574

Experiment 02 weighting:
  0.1 × L_adv + 1.0 × L_semantic

Update verification:
  Generator changed     : True
  Discriminator changed : True
  Baseline unchanged    : True

Gradient verification:
  Generator gradients finite     : True
  Discriminator gradients finite : True

All losses finite : True

SINGLE-BATCH TRAINING TEST PASSED


In [18]:
# ============================================================
# Cell 17: Clean Reset Before Official Experiment 02 Training
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — CLEAN TRAINING RESET")
print("=" * 70)

# ------------------------------------------------------------
# Reset all random seeds
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Recreate Generator from fresh initialization
# ------------------------------------------------------------

generator = Generator().to(device)

# ------------------------------------------------------------
# Recreate Discriminator from fresh initialization
# ------------------------------------------------------------

discriminator = Discriminator().to(device)

# ------------------------------------------------------------
# Recreate and reload the official baseline
# ------------------------------------------------------------

baseline_model = BaselineCrossModalModel().to(device)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True
)

# Freeze baseline
for parameter in baseline_model.parameters():
    parameter.requires_grad = False

baseline_model.eval()

# ------------------------------------------------------------
# Recreate optimizers
# ------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=GAN_LEARNING_RATE
)

discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=GAN_LEARNING_RATE
)

# ------------------------------------------------------------
# Reset DataLoader shuffle generator
# ------------------------------------------------------------

train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

# Recreate training DataLoader with reset shuffle state
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator,
    drop_last=False
)

# Validation remains deterministic and unshuffled
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

# ------------------------------------------------------------
# Put models in correct modes
# ------------------------------------------------------------

generator.train()
discriminator.train()
baseline_model.eval()

# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

generator_trainable = sum(
    parameter.numel()
    for parameter in generator.parameters()
    if parameter.requires_grad
)

discriminator_trainable = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
    if parameter.requires_grad
)

baseline_trainable = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

# ------------------------------------------------------------
# Verify clean state
# ------------------------------------------------------------

assert generator_trainable == 20_768
assert discriminator_trainable == 4_225
assert baseline_trainable == 0

assert generator.training is True
assert discriminator.training is True
assert baseline_model.training is False

assert len(train_loader) == 56
assert len(validation_loader) == 14

# ------------------------------------------------------------
# Display reset status
# ------------------------------------------------------------

print("\nFresh model initialization completed.")

print("\nTraining configuration:")
print(f"  Seed                  : {SEED}")
print(f"  Batch size            : {BATCH_SIZE}")
print(f"  Learning rate         : {GAN_LEARNING_RATE}")
print(f"  Adversarial weight    : {ADVERSARIAL_LOSS_WEIGHT}")
print(f"  Semantic weight       : {SEMANTIC_LOSS_WEIGHT}")

print("\nTrainable parameters:")
print(f"  Generator             : {generator_trainable:,}")
print(f"  Discriminator         : {discriminator_trainable:,}")
print(f"  Baseline              : {baseline_trainable:,}")

print("\nModel modes:")
print(f"  Generator training    : {generator.training}")
print(f"  Discriminator training: {discriminator.training}")
print(f"  Baseline evaluation   : {not baseline_model.training}")

print("\nDataLoader state:")
print(f"  Training batches      : {len(train_loader)}")
print(f"  Validation batches    : {len(validation_loader)}")
print(f"  Training shuffle seed : {SEED}")

print("\n" + "=" * 70)
print("CLEAN RESET PASSED")
print("Ready for OFFICIAL Experiment 02 training.")
print("=" * 70)

EXPERIMENT 02 — CLEAN TRAINING RESET

Fresh model initialization completed.

Training configuration:
  Seed                  : 42
  Batch size            : 32
  Learning rate         : 0.001
  Adversarial weight    : 0.1
  Semantic weight       : 1.0

Trainable parameters:
  Generator             : 20,768
  Discriminator         : 4,225
  Baseline              : 0

Model modes:
  Generator training    : True
  Discriminator training: True
  Baseline evaluation   : True

DataLoader state:
  Training batches      : 56
  Validation batches    : 14
  Training shuffle seed : 42

CLEAN RESET PASSED
Ready for OFFICIAL Experiment 02 training.


In [19]:
# ============================================================
# Cell 18: Official Experiment 02 Training Configuration
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — OFFICIAL TRAINING CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Official training configuration
# ------------------------------------------------------------

NUM_EPOCHS = 50

BASELINE_REFERENCE_ROC_AUC = 0.662667

training_config = {
    "phase": "03.06 GAN Training",
    "experiment": "Experiment 02",
    "experiment_name": "lambda_adv_0.1_lambda_semantic_1.0",

    "purpose": (
        "Controlled experiment reducing adversarial loss "
        "contribution while keeping semantic loss unchanged."
    ),

    "hypothesis": (
        "Reducing the adversarial loss weight from 1.0 to 0.1 "
        "may reduce interference with semantic relationship "
        "preservation."
    ),

    "dataset": {
        "source": "Phase 02.08 representation-aware split",
        "training_pairs": 1776,
        "validation_pairs": 448,
        "embedding_dimension": 32,
        "training_positive": 888,
        "training_negative": 888,
        "validation_positive": 224,
        "validation_negative": 224,
        "training_type": "unpaired_class_consistent_cross_modal_pairs"
    },

    "generator": {
        "architecture": "32-64-128-64-32",
        "parameters": 20768
    },

    "discriminator": {
        "architecture": "32-64-32-1",
        "parameters": 4225
    },

    "frozen_baseline": {
        "architecture": "128-128-64-1",
        "parameters": 24833,
        "trainable_parameters": 0,
        "reference_roc_auc": BASELINE_REFERENCE_ROC_AUC
    },

    "training": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": GAN_LEARNING_RATE,
        "seed": SEED
    },

    "loss": {
        "adversarial_loss": "BCEWithLogitsLoss",
        "semantic_loss": "BCEWithLogitsLoss",
        "adversarial_weight": ADVERSARIAL_LOSS_WEIGHT,
        "semantic_weight": SEMANTIC_LOSS_WEIGHT,
        "generator_objective": (
            "L_G = 0.1 * L_adv + 1.0 * L_semantic"
        )
    },

    "model_selection": {
        "primary_metric": "validation_semantic_roc_auc",
        "baseline_comparison": True
    },

    "controlled_change_from_experiment_01": {
        "adversarial_weight": "1.0 -> 0.1",
        "semantic_weight": "1.0 -> 1.0",
        "architecture_changed": False,
        "dataset_changed": False,
        "optimizer_changed": False,
        "learning_rate_changed": False,
        "batch_size_changed": False,
        "seed_changed": False
    }
}

# ------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------

with open(configuration_path, "w", encoding="utf-8") as file:
    json.dump(
        training_config,
        file,
        indent=4
    )

# ------------------------------------------------------------
# Verify configuration file
# ------------------------------------------------------------

assert configuration_path.exists(), (
    "Experiment 02 configuration file was not created."
)

print("\nOfficial training configuration:")

print(f"  Experiment             : Experiment 02")
print(f"  Epochs                 : {NUM_EPOCHS}")
print(f"  Batch size             : {BATCH_SIZE}")
print(f"  Learning rate          : {GAN_LEARNING_RATE}")
print(f"  Seed                   : {SEED}")

print("\nLoss configuration:")
print(f"  Adversarial weight     : {ADVERSARIAL_LOSS_WEIGHT}")
print(f"  Semantic weight        : {SEMANTIC_LOSS_WEIGHT}")

print("\nModel-selection metric:")
print("  Validation semantic ROC-AUC")

print("\nBaseline reference:")
print(f"  ROC-AUC                : {BASELINE_REFERENCE_ROC_AUC}")

print("\nConfiguration saved to:")
print(f"  {configuration_path}")

print("\n" + "=" * 70)
print("OFFICIAL CONFIGURATION SAVED SUCCESSFULLY")
print("=" * 70)

EXPERIMENT 02 — OFFICIAL TRAINING CONFIGURATION

Official training configuration:
  Experiment             : Experiment 02
  Epochs                 : 50
  Batch size             : 32
  Learning rate          : 0.001
  Seed                   : 42

Loss configuration:
  Adversarial weight     : 0.1
  Semantic weight        : 1.0

Model-selection metric:
  Validation semantic ROC-AUC

Baseline reference:
  ROC-AUC                : 0.662667

Configuration saved to:
  C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_02_lambda_adv_0_1\03_06_gan_experiment_02_configuration.json

OFFICIAL CONFIGURATION SAVED SUCCESSFULLY


In [20]:
# ============================================================
# Cell 19: Final Pre-Training Safety Audit
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — FINAL PRE-TRAINING SAFETY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Dataset checks
# ------------------------------------------------------------

assert len(train_dataset) == 1776
assert len(validation_dataset) == 448

assert train_source.shape == (1776, 32)
assert train_target.shape == (1776, 32)
assert validation_source.shape == (448, 32)
assert validation_target.shape == (448, 32)

# ------------------------------------------------------------
# DataLoader checks
# ------------------------------------------------------------

assert BATCH_SIZE == 32
assert len(train_loader) == 56
assert len(validation_loader) == 14

# ------------------------------------------------------------
# Model parameter checks
# ------------------------------------------------------------

generator_params = sum(
    parameter.numel()
    for parameter in generator.parameters()
)

discriminator_params = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
)

baseline_params = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)

generator_trainable = sum(
    parameter.numel()
    for parameter in generator.parameters()
    if parameter.requires_grad
)

discriminator_trainable = sum(
    parameter.numel()
    for parameter in discriminator.parameters()
    if parameter.requires_grad
)

baseline_trainable = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

assert generator_params == 20_768
assert discriminator_params == 4_225
assert baseline_params == 24_833

assert generator_trainable == 20_768
assert discriminator_trainable == 4_225
assert baseline_trainable == 0

# ------------------------------------------------------------
# Model mode checks
# ------------------------------------------------------------

assert generator.training is True
assert discriminator.training is True
assert baseline_model.training is False

# ------------------------------------------------------------
# Loss configuration checks
# ------------------------------------------------------------

assert ADVERSARIAL_LOSS_WEIGHT == 0.1
assert SEMANTIC_LOSS_WEIGHT == 1.0

assert GAN_LEARNING_RATE == 0.001
assert NUM_EPOCHS == 50
assert SEED == 42

# ------------------------------------------------------------
# Optimizer checks
# ------------------------------------------------------------

generator_optimizer_lr = (
    generator_optimizer.param_groups[0]["lr"]
)

discriminator_optimizer_lr = (
    discriminator_optimizer.param_groups[0]["lr"]
)

assert generator_optimizer_lr == 0.001
assert discriminator_optimizer_lr == 0.001

# ------------------------------------------------------------
# Baseline checkpoint and configuration checks
# ------------------------------------------------------------

assert baseline_checkpoint_path.exists()
assert configuration_path.exists()

# ------------------------------------------------------------
# Experiment 2 output directories
# ------------------------------------------------------------

assert experiment_02_checkpoint_dir.exists()
assert experiment_02_report_dir.exists()
assert experiment_02_figure_dir.exists()

# ------------------------------------------------------------
# Safety: ensure Experiment 1 checkpoint is not the
# Experiment 2 checkpoint directory
# ------------------------------------------------------------

experiment_01_checkpoint_dir = (
    gan_training_root / "checkpoints"
)

assert experiment_01_checkpoint_dir != (
    experiment_02_checkpoint_dir
)

# ------------------------------------------------------------
# Display final audit
# ------------------------------------------------------------

print("\nDataset:")
print("  Training pairs        : 1,776")
print("  Validation pairs      : 448")
print("  Embedding dimension   : 32")

print("\nModels:")
print(f"  Generator parameters  : {generator_params:,}")
print(f"  Discriminator params  : {discriminator_params:,}")
print(f"  Baseline parameters   : {baseline_params:,}")

print("\nTrainable parameters:")
print(f"  Generator             : {generator_trainable:,}")
print(f"  Discriminator         : {discriminator_trainable:,}")
print(f"  Baseline              : {baseline_trainable:,}")

print("\nTraining:")
print(f"  Epochs                : {NUM_EPOCHS}")
print(f"  Batch size            : {BATCH_SIZE}")
print(f"  Learning rate         : {GAN_LEARNING_RATE}")
print(f"  Seed                  : {SEED}")

print("\nLoss:")
print(f"  Adversarial weight    : {ADVERSARIAL_LOSS_WEIGHT}")
print(f"  Semantic weight       : {SEMANTIC_LOSS_WEIGHT}")

print("\nModel selection:")
print("  Primary metric        : Validation semantic ROC-AUC")
print(f"  Baseline reference    : {BASELINE_REFERENCE_ROC_AUC:.6f}")

print("\nOutput isolation:")
print(f"  Experiment 2 root     : {experiment_02_root}")
print(f"  Experiment 1 outputs  : {experiment_01_checkpoint_dir}")
print(f"  Experiment 2 outputs  : {experiment_02_checkpoint_dir}")

print("\n" + "=" * 70)
print("FINAL PRE-TRAINING SAFETY AUDIT PASSED")
print("=" * 70)
print("✓ Dataset verified")
print("✓ DataLoaders verified")
print("✓ Generator verified")
print("✓ Discriminator verified")
print("✓ Baseline frozen")
print("✓ Loss weights verified")
print("✓ Optimizers verified")
print("✓ Checkpoint paths isolated")
print("✓ Experiment 2 is ready for official training")
print("=" * 70)

EXPERIMENT 02 — FINAL PRE-TRAINING SAFETY AUDIT

Dataset:
  Training pairs        : 1,776
  Validation pairs      : 448
  Embedding dimension   : 32

Models:
  Generator parameters  : 20,768
  Discriminator params  : 4,225
  Baseline parameters   : 24,833

Trainable parameters:
  Generator             : 20,768
  Discriminator         : 4,225
  Baseline              : 0

Training:
  Epochs                : 50
  Batch size            : 32
  Learning rate         : 0.001
  Seed                  : 42

Loss:
  Adversarial weight    : 0.1
  Semantic weight       : 1.0

Model selection:
  Primary metric        : Validation semantic ROC-AUC
  Baseline reference    : 0.662667

Output isolation:
  Experiment 2 root     : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_02_lambda_adv_0_1
  Experiment 1 outputs  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_le

In [21]:
# ============================================================
# Cell 20: OFFICIAL EXPERIMENT 02 GAN TRAINING
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — OFFICIAL GAN TRAINING")
print("=" * 70)

print("\nTraining objective:")
print(
    "  L_G = "
    f"{ADVERSARIAL_LOSS_WEIGHT} × L_adv + "
    f"{SEMANTIC_LOSS_WEIGHT} × L_semantic"
)

print("\nConfiguration:")
print(f"  Epochs             : {NUM_EPOCHS}")
print(f"  Batch size         : {BATCH_SIZE}")
print(f"  Learning rate      : {GAN_LEARNING_RATE}")
print(f"  Adversarial weight : {ADVERSARIAL_LOSS_WEIGHT}")
print(f"  Semantic weight    : {SEMANTIC_LOSS_WEIGHT}")
print(f"  Seed               : {SEED}")
print(f"  Device             : {device}")

print("\n" + "-" * 70)

# ------------------------------------------------------------
# Reproducibility reset immediately before official training
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

training_history = {
    "epoch": [],

    # Generator losses
    "generator_total_loss": [],
    "generator_adversarial_loss": [],
    "generator_semantic_loss": [],

    # Discriminator losses
    "discriminator_total_loss": [],
    "discriminator_real_loss": [],
    "discriminator_fake_loss": [],

    # Validation metrics
    "validation_loss": [],
    "validation_accuracy": [],
    "validation_balanced_accuracy": [],
    "validation_precision": [],
    "validation_recall": [],
    "validation_f1": [],
    "validation_roc_auc": []
}

# ------------------------------------------------------------
# Best-model tracking
# ------------------------------------------------------------

best_val_roc_auc = -np.inf
best_epoch = None

training_start_time = time.time()

# ------------------------------------------------------------
# Official training loop
# ------------------------------------------------------------

for epoch in range(1, NUM_EPOCHS + 1):

    generator.train()
    discriminator.train()
    baseline_model.eval()

    # --------------------------------------------------------
    # Reset epoch losses
    # --------------------------------------------------------

    epoch_generator_total = 0.0
    epoch_generator_adversarial = 0.0
    epoch_generator_semantic = 0.0

    epoch_discriminator_total = 0.0
    epoch_discriminator_real = 0.0
    epoch_discriminator_fake = 0.0

    # --------------------------------------------------------
    # Training batches
    # --------------------------------------------------------

    for (
        source,
        real_target,
        labels
    ) in train_loader:

        source = source.to(device)
        real_target = real_target.to(device)
        labels = labels.to(device)

        # ====================================================
        # DISCRIMINATOR UPDATE
        # ====================================================

        discriminator_optimizer.zero_grad()

        # Real target
        real_logits = discriminator(real_target)

        real_labels = torch.ones_like(real_logits)

        discriminator_real_loss = adversarial_loss_fn(
            real_logits,
            real_labels
        )

        # Generated target
        generated_target = generator(source)

        fake_logits = discriminator(
            generated_target.detach()
        )

        fake_labels = torch.zeros_like(fake_logits)

        discriminator_fake_loss = adversarial_loss_fn(
            fake_logits,
            fake_labels
        )

        discriminator_loss = (
            discriminator_real_loss +
            discriminator_fake_loss
        )

        discriminator_loss.backward()

        discriminator_optimizer.step()

        # ====================================================
        # GENERATOR UPDATE
        # ====================================================

        generator_optimizer.zero_grad()

        # Generate target representation
        generated_target = generator(source)

        # ----------------------------------------------------
        # Adversarial Generator loss
        # ----------------------------------------------------

        generated_logits = discriminator(
            generated_target
        )

        generator_real_labels = torch.ones_like(
            generated_logits
        )

        generator_adversarial_loss = adversarial_loss_fn(
            generated_logits,
            generator_real_labels
        )

        # ----------------------------------------------------
        # Semantic relationship loss
        # ----------------------------------------------------

        semantic_logits = baseline_model(
            source,
            generated_target
        )

        generator_semantic_loss = semantic_loss_fn(
            semantic_logits,
            labels
        )

        # ----------------------------------------------------
        # Experiment 02 weighted Generator objective
        # ----------------------------------------------------

        generator_loss = (
            ADVERSARIAL_LOSS_WEIGHT *
            generator_adversarial_loss
            +
            SEMANTIC_LOSS_WEIGHT *
            generator_semantic_loss
        )

        generator_loss.backward()

        generator_optimizer.step()

        # ====================================================
        # Accumulate losses
        # ====================================================

        epoch_generator_total += (
            generator_loss.item()
        )

        epoch_generator_adversarial += (
            generator_adversarial_loss.item()
        )

        epoch_generator_semantic += (
            generator_semantic_loss.item()
        )

        epoch_discriminator_total += (
            discriminator_loss.item()
        )

        epoch_discriminator_real += (
            discriminator_real_loss.item()
        )

        epoch_discriminator_fake += (
            discriminator_fake_loss.item()
        )

    # --------------------------------------------------------
    # Average training losses
    # --------------------------------------------------------

    num_train_batches = len(train_loader)

    avg_generator_total = (
        epoch_generator_total / num_train_batches
    )

    avg_generator_adversarial = (
        epoch_generator_adversarial / num_train_batches
    )

    avg_generator_semantic = (
        epoch_generator_semantic / num_train_batches
    )

    avg_discriminator_total = (
        epoch_discriminator_total / num_train_batches
    )

    avg_discriminator_real = (
        epoch_discriminator_real / num_train_batches
    )

    avg_discriminator_fake = (
        epoch_discriminator_fake / num_train_batches
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    generator.eval()
    discriminator.eval()
    baseline_model.eval()

    validation_losses = []

    all_validation_labels = []
    all_validation_probabilities = []

    with torch.no_grad():

        for (
            source,
            real_target,
            labels
        ) in validation_loader:

            source = source.to(device)
            real_target = real_target.to(device)
            labels = labels.to(device)

            # Generate target representation
            generated_target = generator(source)

            # Semantic relationship prediction
            semantic_logits = baseline_model(
                source,
                generated_target
            )

            semantic_loss = semantic_loss_fn(
                semantic_logits,
                labels
            )

            validation_losses.append(
                semantic_loss.item()
            )

            probabilities = torch.sigmoid(
                semantic_logits
            )

            all_validation_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_validation_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    all_validation_labels = np.asarray(
        all_validation_labels
    )

    all_validation_probabilities = np.asarray(
        all_validation_probabilities
    )

    validation_predictions = (
        all_validation_probabilities >= 0.5
    ).astype(int)

    validation_loss = float(
        np.mean(validation_losses)
    )

    validation_accuracy = accuracy_score(
        all_validation_labels,
        validation_predictions
    )

    validation_balanced_accuracy = (
        balanced_accuracy_score(
            all_validation_labels,
            validation_predictions
        )
    )

    validation_precision = precision_score(
        all_validation_labels,
        validation_predictions,
        zero_division=0
    )

    validation_recall = recall_score(
        all_validation_labels,
        validation_predictions,
        zero_division=0
    )

    validation_f1 = f1_score(
        all_validation_labels,
        validation_predictions,
        zero_division=0
    )

    # ROC-AUC requires both classes
    if len(np.unique(all_validation_labels)) == 2:

        validation_roc_auc = roc_auc_score(
            all_validation_labels,
            all_validation_probabilities
        )

    else:

        validation_roc_auc = float("nan")

    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    training_history["epoch"].append(epoch)

    training_history[
        "generator_total_loss"
    ].append(avg_generator_total)

    training_history[
        "generator_adversarial_loss"
    ].append(avg_generator_adversarial)

    training_history[
        "generator_semantic_loss"
    ].append(avg_generator_semantic)

    training_history[
        "discriminator_total_loss"
    ].append(avg_discriminator_total)

    training_history[
        "discriminator_real_loss"
    ].append(avg_discriminator_real)

    training_history[
        "discriminator_fake_loss"
    ].append(avg_discriminator_fake)

    training_history[
        "validation_loss"
    ].append(validation_loss)

    training_history[
        "validation_accuracy"
    ].append(validation_accuracy)

    training_history[
        "validation_balanced_accuracy"
    ].append(validation_balanced_accuracy)

    training_history[
        "validation_precision"
    ].append(validation_precision)

    training_history[
        "validation_recall"
    ].append(validation_recall)

    training_history[
        "validation_f1"
    ].append(validation_f1)

    training_history[
        "validation_roc_auc"
    ].append(validation_roc_auc)

    # --------------------------------------------------------
    # Check for best model
    # --------------------------------------------------------

    is_best = False

    if np.isfinite(validation_roc_auc):

        if validation_roc_auc > best_val_roc_auc:

            best_val_roc_auc = validation_roc_auc
            best_epoch = epoch
            is_best = True

            # Save best Generator
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": generator.state_dict(),
                    "validation_roc_auc": validation_roc_auc,
                    "adversarial_loss_weight":
                        ADVERSARIAL_LOSS_WEIGHT,
                    "semantic_loss_weight":
                        SEMANTIC_LOSS_WEIGHT,
                    "seed": SEED
                },
                best_generator_path
            )

            # Save best Discriminator
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": discriminator.state_dict(),
                    "validation_roc_auc": validation_roc_auc,
                    "adversarial_loss_weight":
                        ADVERSARIAL_LOSS_WEIGHT,
                    "semantic_loss_weight":
                        SEMANTIC_LOSS_WEIGHT,
                    "seed": SEED
                },
                best_discriminator_path
            )

    # --------------------------------------------------------
    # Print epoch results
    # --------------------------------------------------------

    best_marker = " ← BEST" if is_best else ""

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"G: {avg_generator_total:.4f} | "
        f"D: {avg_discriminator_total:.4f} | "
        f"Sem: {avg_generator_semantic:.4f} | "
        f"Val Loss: {validation_loss:.4f} | "
        f"Val AUC: {validation_roc_auc:.4f}"
        f"{best_marker}"
    )

# ============================================================
# SAVE FINAL MODELS
# ============================================================

torch.save(
    {
        "epoch": NUM_EPOCHS,
        "model_state_dict": generator.state_dict(),
        "seed": SEED,
        "adversarial_loss_weight":
            ADVERSARIAL_LOSS_WEIGHT,
        "semantic_loss_weight":
            SEMANTIC_LOSS_WEIGHT
    },
    final_generator_path
)

torch.save(
    {
        "epoch": NUM_EPOCHS,
        "model_state_dict": discriminator.state_dict(),
        "seed": SEED,
        "adversarial_loss_weight":
            ADVERSARIAL_LOSS_WEIGHT,
        "semantic_loss_weight":
            SEMANTIC_LOSS_WEIGHT
    },
    final_discriminator_path
)

# ------------------------------------------------------------
# Save training history
# ------------------------------------------------------------

torch.save(
    training_history,
    training_history_path
)

# ============================================================
# TRAINING COMPLETION
# ============================================================

training_elapsed = (
    time.time() - training_start_time
)

print("\n" + "=" * 70)
print("EXPERIMENT 02 TRAINING COMPLETED")
print("=" * 70)

print(f"Training time          : {training_elapsed:.2f} seconds")

print(
    f"Best validation AUC    : "
    f"{best_val_roc_auc:.6f}"
)

print(
    f"Best epoch             : "
    f"{best_epoch}"
)

print(
    f"Baseline reference AUC : "
    f"{BASELINE_REFERENCE_ROC_AUC:.6f}"
)

if best_val_roc_auc > BASELINE_REFERENCE_ROC_AUC:
    print("\n✓ GAN exceeded the baseline reference.")

elif best_val_roc_auc > 0.500000:
    print(
        "\n✓ GAN learned above chance, "
        "but did not exceed the baseline."
    )

else:
    print(
        "\n⚠ GAN did not demonstrate "
        "above-chance validation performance."
    )

print("\nSaved artifacts:")
print(f"  Best Generator      : {best_generator_path}")
print(f"  Best Discriminator  : {best_discriminator_path}")
print(f"  Final Generator     : {final_generator_path}")
print(f"  Final Discriminator : {final_discriminator_path}")
print(f"  Training History    : {training_history_path}")

print("\n" + "=" * 70)
print("OFFICIAL EXPERIMENT 02 TRAINING FINISHED")
print("=" * 70)

EXPERIMENT 02 — OFFICIAL GAN TRAINING

Training objective:
  L_G = 0.1 × L_adv + 1.0 × L_semantic

Configuration:
  Epochs             : 50
  Batch size         : 32
  Learning rate      : 0.001
  Adversarial weight : 0.1
  Semantic weight    : 1.0
  Seed               : 42
  Device             : cpu

----------------------------------------------------------------------
Epoch 01/50 | G: 0.7785 | D: 0.8526 | Sem: 0.7096 | Val Loss: 0.6999 | Val AUC: 0.5000 ← BEST
Epoch 02/50 | G: 0.8100 | D: 0.5108 | Sem: 0.6985 | Val Loss: 0.6970 | Val AUC: 0.5000
Epoch 03/50 | G: 0.8351 | D: 0.5184 | Sem: 0.7018 | Val Loss: 0.6999 | Val AUC: 0.5000
Epoch 04/50 | G: 0.9131 | D: 0.3688 | Sem: 0.7053 | Val Loss: 0.7022 | Val AUC: 0.5000
Epoch 05/50 | G: 0.9627 | D: 0.3315 | Sem: 0.7084 | Val Loss: 0.7005 | Val AUC: 0.5000
Epoch 06/50 | G: 0.9647 | D: 0.4412 | Sem: 0.7137 | Val Loss: 0.7006 | Val AUC: 0.5000
Epoch 07/50 | G: 0.9958 | D: 0.3278 | Sem: 0.7196 | Val Loss: 0.6984 | Val AUC: 0.5000
Epoch 08/5

In [22]:
# ============================================================
# Cell 21: Experiment 02 Post-Training Diagnostic
# ============================================================

print("=" * 70)
print("EXPERIMENT 02 — POST-TRAINING DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# Load saved training history
# ------------------------------------------------------------

if not training_history_path.exists():
    raise FileNotFoundError(
        f"Training history not found:\n"
        f"{training_history_path}"
    )

experiment_02_history = torch.load(
    training_history_path,
    map_location="cpu"
)

print("\nTraining history loaded successfully.")

# ------------------------------------------------------------
# Convert history values to NumPy arrays
# ------------------------------------------------------------

epochs = np.asarray(
    experiment_02_history["epoch"],
    dtype=int
)

generator_total = np.asarray(
    experiment_02_history["generator_total_loss"],
    dtype=float
)

generator_adversarial = np.asarray(
    experiment_02_history["generator_adversarial_loss"],
    dtype=float
)

generator_semantic = np.asarray(
    experiment_02_history["generator_semantic_loss"],
    dtype=float
)

discriminator_total = np.asarray(
    experiment_02_history["discriminator_total_loss"],
    dtype=float
)

discriminator_real = np.asarray(
    experiment_02_history["discriminator_real_loss"],
    dtype=float
)

discriminator_fake = np.asarray(
    experiment_02_history["discriminator_fake_loss"],
    dtype=float
)

validation_loss = np.asarray(
    experiment_02_history["validation_loss"],
    dtype=float
)

validation_accuracy = np.asarray(
    experiment_02_history["validation_accuracy"],
    dtype=float
)

validation_balanced_accuracy = np.asarray(
    experiment_02_history["validation_balanced_accuracy"],
    dtype=float
)

validation_precision = np.asarray(
    experiment_02_history["validation_precision"],
    dtype=float
)

validation_recall = np.asarray(
    experiment_02_history["validation_recall"],
    dtype=float
)

validation_f1 = np.asarray(
    experiment_02_history["validation_f1"],
    dtype=float
)

validation_roc_auc = np.asarray(
    experiment_02_history["validation_roc_auc"],
    dtype=float
)

# ------------------------------------------------------------
# Basic history integrity
# ------------------------------------------------------------

assert len(epochs) == NUM_EPOCHS
assert len(validation_roc_auc) == NUM_EPOCHS

assert np.isfinite(generator_total).all()
assert np.isfinite(generator_adversarial).all()
assert np.isfinite(generator_semantic).all()

assert np.isfinite(discriminator_total).all()
assert np.isfinite(discriminator_real).all()
assert np.isfinite(discriminator_fake).all()

assert np.isfinite(validation_loss).all()
assert np.isfinite(validation_roc_auc).all()

print("✓ 50 epochs present in history")
print("✓ All recorded losses are finite")
print("✓ All validation ROC-AUC values are finite")

# ============================================================
# EXACT ROC-AUC ANALYSIS
# ============================================================

best_auc_index = int(
    np.argmax(validation_roc_auc)
)

exact_best_auc = float(
    validation_roc_auc[best_auc_index]
)

exact_best_epoch = int(
    epochs[best_auc_index]
)

minimum_auc = float(
    np.min(validation_roc_auc)
)

maximum_auc = float(
    np.max(validation_roc_auc)
)

mean_auc = float(
    np.mean(validation_roc_auc)
)

std_auc = float(
    np.std(validation_roc_auc)
)

# ------------------------------------------------------------
# Number of distinct AUC values
# ------------------------------------------------------------

unique_auc_values = np.unique(
    validation_roc_auc
)

# ============================================================
# LOSS TREND ANALYSIS
# ============================================================

initial_semantic_loss = float(
    generator_semantic[0]
)

final_semantic_loss = float(
    generator_semantic[-1]
)

minimum_semantic_loss = float(
    np.min(generator_semantic)
)

minimum_semantic_epoch = int(
    epochs[np.argmin(generator_semantic)]
)

initial_validation_loss = float(
    validation_loss[0]
)

final_validation_loss = float(
    validation_loss[-1]
)

minimum_validation_loss = float(
    np.min(validation_loss)
)

minimum_validation_epoch = int(
    epochs[np.argmin(validation_loss)]
)

# ============================================================
# DISCRIMINATOR TREND
# ============================================================

initial_discriminator_loss = float(
    discriminator_total[0]
)

final_discriminator_loss = float(
    discriminator_total[-1]
)

# ============================================================
# GENERATOR LOSS TREND
# ============================================================

initial_generator_total = float(
    generator_total[0]
)

final_generator_total = float(
    generator_total[-1]
)

# ============================================================
# PRINT EXACT RESULTS
# ============================================================

print("\n" + "=" * 70)
print("EXACT VALIDATION ROC-AUC ANALYSIS")
print("=" * 70)

print(
    f"Best validation ROC-AUC : "
    f"{exact_best_auc:.10f}"
)

print(
    f"Best epoch              : "
    f"{exact_best_epoch}"
)

print(
    f"Minimum ROC-AUC         : "
    f"{minimum_auc:.10f}"
)

print(
    f"Maximum ROC-AUC         : "
    f"{maximum_auc:.10f}"
)

print(
    f"Mean ROC-AUC            : "
    f"{mean_auc:.10f}"
)

print(
    f"ROC-AUC standard dev.   : "
    f"{std_auc:.10f}"
)

print(
    f"Unique ROC-AUC values   : "
    f"{len(unique_auc_values)}"
)

print("\nFirst 10 ROC-AUC values:")

for epoch, auc in zip(
    epochs[:10],
    validation_roc_auc[:10]
):
    print(
        f"  Epoch {epoch:02d}: "
        f"{auc:.10f}"
    )

print("\nLast 10 ROC-AUC values:")

for epoch, auc in zip(
    epochs[-10:],
    validation_roc_auc[-10:]
):
    print(
        f"  Epoch {epoch:02d}: "
        f"{auc:.10f}"
    )

# ============================================================
# SEMANTIC LOSS ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("SEMANTIC LOSS ANALYSIS")
print("=" * 70)

print(
    f"Initial semantic loss  : "
    f"{initial_semantic_loss:.10f}"
)

print(
    f"Final semantic loss    : "
    f"{final_semantic_loss:.10f}"
)

print(
    f"Minimum semantic loss  : "
    f"{minimum_semantic_loss:.10f}"
)

print(
    f"Minimum loss epoch     : "
    f"{minimum_semantic_epoch}"
)

print(
    f"Semantic loss change   : "
    f"{final_semantic_loss - initial_semantic_loss:+.10f}"
)

# ============================================================
# VALIDATION LOSS ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION LOSS ANALYSIS")
print("=" * 70)

print(
    f"Initial validation loss : "
    f"{initial_validation_loss:.10f}"
)

print(
    f"Final validation loss   : "
    f"{final_validation_loss:.10f}"
)

print(
    f"Minimum validation loss : "
    f"{minimum_validation_loss:.10f}"
)

print(
    f"Minimum loss epoch      : "
    f"{minimum_validation_epoch}"
)

# ============================================================
# OVERALL TRAINING TREND
# ============================================================

print("\n" + "=" * 70)
print("OVERALL TRAINING TREND")
print("=" * 70)

print(
    f"Generator total loss:"
    f"\n  Initial : {initial_generator_total:.10f}"
    f"\n  Final   : {final_generator_total:.10f}"
)

print(
    f"\nDiscriminator total loss:"
    f"\n  Initial : {initial_discriminator_loss:.10f}"
    f"\n  Final   : {final_discriminator_loss:.10f}"
)

# ============================================================
# COMPARISON WITH EXPERIMENT 01 AND BASELINE
# ============================================================

EXPERIMENT_01_ROC_AUC = 0.5000000000

print("\n" + "=" * 70)
print("COMPARISON WITH PREVIOUS RESULTS")
print("=" * 70)

print(
    f"Baseline ROC-AUC       : "
    f"{BASELINE_REFERENCE_ROC_AUC:.10f}"
)

print(
    f"Experiment 01 ROC-AUC  : "
    f"{EXPERIMENT_01_ROC_AUC:.10f}"
)

print(
    f"Experiment 02 ROC-AUC  : "
    f"{exact_best_auc:.10f}"
)

print(
    f"\nExperiment 02 - "
    f"Experiment 01:"
)

print(
    f"  {exact_best_auc - EXPERIMENT_01_ROC_AUC:+.10f}"
)

print(
    f"\nExperiment 02 - Baseline:"
)

print(
    f"  {exact_best_auc - BASELINE_REFERENCE_ROC_AUC:+.10f}"
)

# ============================================================
# FINAL INTERPRETATION
# ============================================================

print("\n" + "=" * 70)
print("DIAGNOSTIC INTERPRETATION")
print("=" * 70)

if exact_best_auc > EXPERIMENT_01_ROC_AUC + 1e-6:
    print(
        "✓ Experiment 02 improved over Experiment 01."
    )
else:
    print(
        "⚠ Experiment 02 did not improve over Experiment 01."
    )

if exact_best_auc >= BASELINE_REFERENCE_ROC_AUC:
    print(
        "✓ Experiment 02 matched or exceeded the baseline."
    )
else:
    print(
        "⚠ Experiment 02 remained below the baseline."
    )

if len(unique_auc_values) == 1:
    print(
        "⚠ Validation ROC-AUC is exactly constant "
        "across all epochs."
    )
else:
    print(
        f"✓ Validation ROC-AUC contains "
        f"{len(unique_auc_values)} distinct values."
    )

if abs(
    final_semantic_loss - initial_semantic_loss
) < 0.01:
    print(
        "⚠ Semantic loss showed very little "
        "overall improvement."
    )
else:
    print(
        "✓ Semantic loss changed meaningfully "
        "during training."
    )

print("\n" + "=" * 70)
print("EXPERIMENT 02 POST-TRAINING DIAGNOSTIC COMPLETED")
print("=" * 70)

EXPERIMENT 02 — POST-TRAINING DIAGNOSTIC

Training history loaded successfully.
✓ 50 epochs present in history
✓ All recorded losses are finite
✓ All validation ROC-AUC values are finite

EXACT VALIDATION ROC-AUC ANALYSIS
Best validation ROC-AUC : 0.5000000000
Best epoch              : 1
Minimum ROC-AUC         : 0.5000000000
Maximum ROC-AUC         : 0.5000000000
Mean ROC-AUC            : 0.5000000000
ROC-AUC standard dev.   : 0.0000000000
Unique ROC-AUC values   : 3

First 10 ROC-AUC values:
  Epoch 01: 0.5000000000
  Epoch 02: 0.5000000000
  Epoch 03: 0.5000000000
  Epoch 04: 0.5000000000
  Epoch 05: 0.5000000000
  Epoch 06: 0.5000000000
  Epoch 07: 0.5000000000
  Epoch 08: 0.5000000000
  Epoch 09: 0.5000000000
  Epoch 10: 0.5000000000

Last 10 ROC-AUC values:
  Epoch 41: 0.5000000000
  Epoch 42: 0.5000000000
  Epoch 43: 0.5000000000
  Epoch 44: 0.5000000000
  Epoch 45: 0.5000000000
  Epoch 46: 0.5000000000
  Epoch 47: 0.5000000000
  Epoch 48: 0.5000000000
  Epoch 49: 0.5000000000
 